# Moodify – Notebook 1/2: Mô hình tiếng Anh (RoBERTa, 6 cảm xúc)

**Dữ liệu:** `dair-ai/emotion` (câu ngôi thứ nhất kiểu "i feel ...", 6 nhãn: sadness, joy, love, anger, fear, surprise).
**Mục tiêu đo:** accuracy ≥ 85% trên tập test chính thức (2.000 câu) của dataset.
**Đầu ra:** `best_model/`, `label_config.json` (nhãn + điểm neo V/A), `moodify_emotion_en.csv` (dùng lại cho Notebook 2 để dịch sang tiếng Việt).

Chạy trên Google Colab, bật GPU (Runtime → Change runtime type → T4).

In [ ]:
!pip install -q transformers datasets scikit-learn pandas

In [ ]:
import os, re, json, math, random, datetime, logging
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          get_linear_schedule_with_warmup)
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

# ======================= CẤU HÌNH =======================
SEED = 42
MODEL_NAME = "roberta-base"
DRIVE_PATH = "/content/drive/MyDrive/train-KLCN"

MAX_LEN = 64            # tweet ngắn, 64 token là đủ
BATCH_SIZE = 32
EPOCHS = 4
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
PATIENCE = 2            # early stopping theo val accuracy
TARGET_ACCURACY = 0.85  # mức bạn yêu cầu

# Bổ sung câu từ bản "unsplit" (416k câu) cho tới khi mỗi lớp có ~PER_CLASS_TARGET câu train.
# Giúp các lớp hiếm (surprise, love, fear, anger). Đặt 0 để chỉ dùng 16k câu train gốc.
# Lưu ý: Notebook 2 sẽ DỊCH toàn bộ số câu này, nên số càng lớn thì dịch càng lâu.
PER_CLASS_TARGET = 4000

# Thứ tự trùng với dair-ai/emotion: 0 sadness, 1 joy, 2 love, 3 anger, 4 fear, 5 surprise
LABELS = ["Sadness", "Enjoyment", "Love", "Anger", "Fear", "Surprise"]
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL = {i: l for l, i in LABEL2ID.items()}

# Điểm neo Valence/Arousal. 5 nhãn đầu giữ đúng giá trị VA_ANCHORS trong notebook PhoBERT của bạn.
# "Love" là giá trị MỚI do mình đề xuất (V cao, A trung bình) - hãy chỉnh theo vùng mood trong DB.
VA_ANCHORS = {
    "Sadness":   [-0.8, -0.6],
    "Enjoyment": [ 0.8,  0.7],
    "Love":      [ 0.8,  0.3],
    "Anger":     [-0.8,  0.8],
    "Fear":      [-0.7,  0.9],
    "Surprise":  [ 0.5,  0.9],
}
# ========================================================

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    DRIVE_PATH = "./train-KLCN"      # chạy local
os.makedirs(DRIVE_PATH, exist_ok=True)

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = os.path.join(DRIVE_PATH, f"output_en_emotion_{timestamp}")
os.makedirs(OUTPUT_DIR, exist_ok=True)
EN_CSV = os.path.join(DRIVE_PATH, "moodify_emotion_en.csv")

for h in logging.root.handlers[:]:
    logging.root.removeHandler(h)
logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[logging.FileHandler(os.path.join(OUTPUT_DIR, "training_log.txt"), mode="w", encoding="utf-8"),
              logging.StreamHandler()])
for noisy in ("httpx", "httpcore", "huggingface_hub", "urllib3", "filelock"):
    logging.getLogger(noisy).setLevel(logging.WARNING)
logger = logging.getLogger("moodify_en")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda"
logger.info("Device: %s | AMP fp16: %s | Output: %s", device, use_amp, OUTPUT_DIR)
if device.type != "cuda":
    logger.warning("Đang chạy CPU - sẽ rất chậm. Hãy bật GPU trong Colab.")

## 1. Tải và chuẩn bị dữ liệu

- Dùng split gốc 16.000 / 2.000 / 2.000. **Val/test không bị chạm vào.**
- Loại câu train trùng với val/test (tránh rò rỉ).
- Top-up các lớp hiếm từ bản `unsplit`, đã loại mọi câu trùng với val/test.

In [ ]:
from datasets import load_dataset

ds = load_dataset("dair-ai/emotion", "split")
expected = ["sadness", "joy", "love", "anger", "fear", "surprise"]
assert ds["train"].features["label"].names == expected, \
    f"Thứ tự nhãn dataset khác dự kiến: {ds['train'].features['label'].names}"

train_df = ds["train"].to_pandas()
val_df = ds["validation"].to_pandas()
test_df = ds["test"].to_pandas()

def norm_key(s):
    return s.str.lower().str.replace(r"\s+", " ", regex=True).str.strip()

for d in (train_df, val_df, test_df):
    d["text"] = d["text"].astype(str).str.strip()
    d["key"] = norm_key(d["text"])

heldout = set(val_df["key"]) | set(test_df["key"])
n0 = len(train_df)
train_df = train_df[~train_df["key"].isin(heldout)].drop_duplicates("key")
logger.info("Loại %d câu train trùng val/test hoặc trùng nhau.", n0 - len(train_df))

if PER_CLASS_TARGET > 0:
    un = load_dataset("dair-ai/emotion", "unsplit")["train"].to_pandas()
    un["text"] = un["text"].astype(str).str.strip()
    un["key"] = norm_key(un["text"])
    used = heldout | set(train_df["key"])
    un = un[~un["key"].isin(used)].drop_duplicates("key")
    extras = []
    for lab in range(len(LABELS)):
        need = PER_CLASS_TARGET - int((train_df["label"] == lab).sum())
        pool = un[un["label"] == lab]
        if need > 0 and len(pool) > 0:
            extras.append(pool.sample(min(need, len(pool)), random_state=SEED))
    if extras:
        train_df = pd.concat([train_df] + extras, ignore_index=True)

train_df = train_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

all_df = pd.concat([train_df.assign(split="train"),
                    val_df.assign(split="val"),
                    test_df.assign(split="test")], ignore_index=True)[["text", "label", "split"]]
all_df["label_name"] = all_df["label"].map(ID2LABEL)
all_df.to_csv(EN_CSV, index=False, encoding="utf-8-sig")

logger.info("Kích thước: train=%d | val=%d | test=%d", len(train_df), len(val_df), len(test_df))
logger.info("Phân bố nhãn train: %s", train_df["label"].map(ID2LABEL).value_counts().to_dict())
logger.info("Phân bố nhãn test : %s", test_df["label"].map(ID2LABEL).value_counts().to_dict())
logger.info("Đã lưu %s (dùng cho Notebook 2).", EN_CSV)

## 2. Dataset, model, vòng huấn luyện

In [ ]:
class TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts, self.labels = list(texts), [int(x) for x in labels]
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, i):
        return self.texts[i], self.labels[i]

def make_loader(df, tokenizer, shuffle):
    def collate(batch):
        texts, labels = zip(*batch)
        enc = tokenizer(list(texts), padding=True, truncation=True,
                        max_length=MAX_LEN, return_tensors="pt")
        enc["labels"] = torch.tensor(labels, dtype=torch.long)
        return enc
    return DataLoader(TextDataset(df["text"], df["label"]), batch_size=BATCH_SIZE,
                      shuffle=shuffle, collate_fn=collate, num_workers=0)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    ys, ps, confs, loss_sum, n = [], [], [], 0.0, 0
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            out = model(**batch)
        probs = torch.softmax(out.logits.float(), dim=-1)
        conf, pred = probs.max(dim=-1)
        bs = batch["labels"].size(0)
        loss_sum += out.loss.item() * bs; n += bs
        ys += batch["labels"].tolist(); ps += pred.tolist(); confs += conf.tolist()
    return {"loss": loss_sum / max(n, 1),
            "accuracy": accuracy_score(ys, ps),
            "macro_f1": f1_score(ys, ps, average="macro", zero_division=0),
            "labels": ys, "preds": ps, "confs": confs}

def run_training(model, tokenizer, train_loader, val_loader, out_dir):
    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    total_steps = len(train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer, max(1, int(total_steps * WARMUP_RATIO)), total_steps)
    scaler = torch.amp.GradScaler(device.type, enabled=use_amp)
    best_dir = os.path.join(out_dir, "best_model")
    best_acc, best_f1, bad, history = -1.0, -1.0, 0, []

    for epoch in range(1, EPOCHS + 1):
        model.train()
        loss_sum, correct, seen = 0.0, 0, 0
        for step, batch in enumerate(train_loader, 1):
            batch = {k: v.to(device) for k, v in batch.items()}
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                out = model(**batch)
            loss = out.loss
            if not torch.isfinite(loss):
                raise RuntimeError(f"Loss không hữu hạn ở epoch {epoch}, step {step}")
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update(); scheduler.step()

            bs = batch["labels"].size(0)
            loss_sum += loss.item() * bs; seen += bs
            correct += (out.logits.argmax(-1) == batch["labels"]).sum().item()
            if step % 100 == 0 or step == len(train_loader):
                logger.info("Epoch %d/%d step %d/%d | loss %.4f | acc %.4f",
                            epoch, EPOCHS, step, len(train_loader), loss_sum / seen, correct / seen)

        val = evaluate(model, val_loader)
        history.append({"epoch": epoch, "train_loss": loss_sum / seen, "train_acc": correct / seen,
                        "val_loss": val["loss"], "val_acc": val["accuracy"], "val_macro_f1": val["macro_f1"]})
        logger.info("== Epoch %d | val loss %.4f | val acc %.4f | val macro-F1 %.4f",
                    epoch, val["loss"], val["accuracy"], val["macro_f1"])

        improved = val["accuracy"] > best_acc or (
            math.isclose(val["accuracy"], best_acc, abs_tol=1e-12) and val["macro_f1"] > best_f1)
        if improved:
            best_acc, best_f1, bad = val["accuracy"], val["macro_f1"], 0
            model.save_pretrained(best_dir); tokenizer.save_pretrained(best_dir)
            logger.info("Lưu checkpoint tốt nhất (epoch %d, val acc %.4f)", epoch, best_acc)
        else:
            bad += 1
            if bad >= PATIENCE:
                logger.info("Early stopping sau epoch %d.", epoch)
                break
        with open(os.path.join(out_dir, "training_metrics.json"), "w", encoding="utf-8") as f:
            json.dump(history, f, ensure_ascii=False, indent=2)
    return best_dir, history

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
train_loader = make_loader(train_df, tokenizer, shuffle=True)
val_loader = make_loader(val_df, tokenizer, shuffle=False)
test_loader = make_loader(test_df, tokenizer, shuffle=False)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(LABELS), id2label=ID2LABEL, label2id=LABEL2ID).to(device)

best_dir, history = run_training(model, tokenizer, train_loader, val_loader, OUTPUT_DIR)
logger.info("Train xong. Best checkpoint: %s", best_dir)

## 3. Đánh giá trên test chính thức (chỉ chạy 1 lần, sau khi train xong)

In [ ]:
del model
if torch.cuda.is_available():
    torch.cuda.empty_cache()
model = AutoModelForSequenceClassification.from_pretrained(best_dir).to(device)
tokenizer = AutoTokenizer.from_pretrained(best_dir)
model.eval()

test_res = evaluate(model, make_loader(test_df, tokenizer, shuffle=False))
acc, f1 = test_res["accuracy"], test_res["macro_f1"]
print(f"\nTEST accuracy = {acc:.2%} | macro-F1 = {f1:.4f}  ->  "
      f"{'ĐẠT' if acc >= TARGET_ACCURACY else 'CHƯA ĐẠT'} mục tiêu {TARGET_ACCURACY:.0%}")

print("\n=== Classification report (test) ===")
print(classification_report(test_res["labels"], test_res["preds"], labels=list(range(len(LABELS))),
                            target_names=LABELS, digits=4, zero_division=0))
cm = pd.DataFrame(confusion_matrix(test_res["labels"], test_res["preds"], labels=list(range(len(LABELS)))),
                  index=LABELS, columns=LABELS)
print("=== Confusion matrix (hàng = đúng, cột = dự đoán) ===")
print(cm)
cm.to_csv(os.path.join(OUTPUT_DIR, "test_confusion_matrix.csv"), encoding="utf-8-sig")

## 4. Hàm dự đoán cho service + lưu cấu hình nhãn / điểm neo V/A

`predict_emotion` trả về cùng dạng kết quả với nhánh tiếng Việt (nhãn, confidence, V/A, xác suất từng lớp), nên có thể cắm vào route `/predict-va` mà backend .NET không phải đổi hợp đồng.
V/A ở đây = Σ xác suất × điểm neo, **không phải** nhãn VA thật.

In [ ]:
def predict_emotion(text, threshold=None):
    if not isinstance(text, str) or not text.strip():
        raise ValueError("Văn bản không được để trống.")
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_LEN).to(device)
    model.eval()
    with torch.no_grad():
        probs = torch.softmax(model(**enc).logits.float(), dim=-1)[0].cpu().numpy()
    k = int(np.argmax(probs))
    valence = float(sum(probs[i] * VA_ANCHORS[LABELS[i]][0] for i in range(len(LABELS))))
    arousal = float(sum(probs[i] * VA_ANCHORS[LABELS[i]][1] for i in range(len(LABELS))))
    return {"emotion": LABELS[k], "confidence": round(float(probs[k]), 4),
            "accepted": None if threshold is None else bool(probs[k] >= threshold),
            "valence": round(valence, 4), "arousal": round(arousal, 4),
            "probabilities": {LABELS[i]: round(float(probs[i]), 4) for i in range(len(LABELS))}}

with open(os.path.join(OUTPUT_DIR, "label_config.json"), "w", encoding="utf-8") as f:
    json.dump({"labels": LABELS, "id2label": ID2LABEL, "va_anchors": VA_ANCHORS,
               "model_name": MODEL_NAME, "max_len": MAX_LEN,
               "test_accuracy": acc, "test_macro_f1": f1,
               "test_set": "dair-ai/emotion split=test (2000 câu, in-distribution)"},
              f, ensure_ascii=False, indent=2)

# --- Kiểm tra nhanh bằng câu do mình tự viết (KHÔNG phải gold set, chỉ để thấy domain shift) ---
SMOKE_EN = [
    ("I haven't talked to anyone in days and the apartment feels huge", "Sadness"),
    ("My heart is pounding, I think someone is following me", "Fear"),
    ("I can't stop smiling, she said yes", "Enjoyment"),
    ("Wait, what? You got the job? No way", "Surprise"),
    ("I would do anything for him, he means the world to me", "Love"),
    ("I'm so sick of being ignored at work, it's completely unfair", "Anger"),
    ("I miss my grandmother so much it hurts", "Sadness"),
    ("The exam is tomorrow and I'm terrified I'll blank out", "Fear"),
    ("Best day ever, we won the whole tournament", "Enjoyment"),
    ("I never expected to see you here, what a shock", "Surprise"),
    ("She is the most precious person in my life", "Love"),
    ("How dare they talk to me like that, I'm furious", "Anger"),
]
hit = 0
for text, gold in SMOKE_EN:
    r = predict_emotion(text)
    hit += int(r["emotion"] == gold)
    print(f"[{'OK ' if r['emotion'] == gold else 'XX '}] gold={gold:9s} pred={r['emotion']:9s} conf={r['confidence']:.2f} | {text}")
print(f"\nSmoke test: {hit}/{len(SMOKE_EN)} đúng (câu tự viết, chỉ mang tính tham khảo)")

print("\n--- Câu trung tính (không có nhãn đúng; xem confidence để chọn ngưỡng 'accepted') ---")
for text in ["I had rice and vegetables for lunch today", "The meeting is at three o'clock", "I'm just relaxing with a cup of tea"]:
    r = predict_emotion(text)
    print(f"pred={r['emotion']:9s} conf={r['confidence']:.2f} V={r['valence']:+.2f} A={r['arousal']:+.2f} | {text}")
print("\nArtifact nằm tại:", OUTPUT_DIR)